# Scalpel — reproducible demo

This notebook regenerates **every plot and the results table** in the README
end to end: SAE reconstruction sanity → feature discovery → qualitative
steering → the coefficient sweep with **baseline controls** and a
**specificity** panel.

It runs on CPU with `gpt2-small` + the released `gpt2-small-res-jb` SAE
(downloaded and cached on first run). Requires the model + viz extras:

```bash
uv pip install -e '.[models,viz,judge]'
```

Set `USE_REAL = False` for a zero-download dry run on the mock backend
(the pipeline executes, but the numbers are only meaningful with the real model).

In [ ]:
# Run from the repo root regardless of where the kernel started.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

USE_REAL = True  # False -> mock backend, no downloads (trivial numbers)

FEATURE = 8243          # 'references to dogs' (Neuronpedia label)
CONCEPT = 'dog'
TERMS = ['dog', 'dogs', 'puppy', 'puppies']
PROBES = ['cat', 'ocean', 'music']
COEFS = [-40, -20, -10, 0, 10, 20, 30, 40, 50, 60]
PROMPTS = [
    'I think that', 'The weather today', 'Let me tell you about',
    'My favorite thing is', 'Yesterday I went to', 'When I woke up this morning',
]
OUT = 'outputs/demo'

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from scalpel.config import load_config, mock_config
from scalpel.seed import set_seed
from scalpel.backends import build_backend
from scalpel.cli import build_sae

cfg = load_config('configs/gpt2-small.yaml') if USE_REAL else mock_config()
set_seed(cfg.seed)
backend = build_backend(cfg)
sae = build_sae(cfg, backend)
print('model :', cfg.model.name, '| device:', backend.device)
print('hook  :', sae.hook_name, '| d_model/d_sae:', sae.d_model, '/', sae.d_sae)

## 1. SAE reconstruction sanity

The SAE should reconstruct the residual stream well (high variance explained)
with a sparse code (low L0).

In [ ]:
sample = [
    'The quick brown fox jumps over the lazy dog.',
    'Paris is the capital of France, and it sits on the Seine.',
]
acts = backend.capture_resid(sample, sae.hook_name)
stats = sae.reconstruction_error(acts)
print(f'variance explained: {stats.variance_explained:.4f}')
print(f'reconstruction MSE: {stats.mse:.4f}')
print(f'mean L0           : {stats.mean_l0:.1f}')

## 2. Feature discovery

Find the SAE features most associated with the concept via contrastive
scoring over a corpus, and (optionally) fetch their Neuronpedia labels.

In [ ]:
from scalpel.corpus import load_corpus
from scalpel.discovery import discover_features
from scalpel.neuronpedia import fetch_label

snippets = load_corpus()  # bundled sample corpus
disc = discover_features(backend, sae, snippets, CONCEPT, terms=TERMS, top_k=5)
for rank, hit in enumerate(disc.hits, 1):
    label = fetch_label(disc.neuronpedia_id, hit.index) if (USE_REAL and disc.neuronpedia_id) else None
    print(f'#{rank} feature {hit.index:>6}  score={hit.score:+.2f}  {label or ""}')

## 3. Qualitative steering (before / after)

Add `coef * W_dec[FEATURE]` to the residual during greedy decoding — the only
difference between the two completions is the steering vector.

In [ ]:
COEF_DEMO = 50 if USE_REAL else 5
prompt = 'My favorite thing to do on the weekend is'
vector = sae.feature_direction(FEATURE if USE_REAL else 1)

set_seed(cfg.seed)
base = backend.generate(prompt, max_new_tokens=40, hook_name=sae.hook_name)
set_seed(cfg.seed)
steered = backend.generate(prompt, max_new_tokens=40, hook_name=sae.hook_name, vector=vector, coef=COEF_DEMO)
print('--- unsteered ---'); print(base)
print('\n--- steered ---'); print(steered)

## 4. Dose-response, fluency, baselines, specificity

One sweep produces it all: effect (keyword scorer), fluency (perplexity + KL),
the **random** and **mean-difference** baseline controls, and the specificity
probes.

In [ ]:
from scalpel.experiment import run_sweep, build_meandiff_direction
from scalpel.metrics.effect import KeywordScorer
from scalpel.steering import build_random_vector, match_norm
from scalpel.discovery import label_snippets

feat = FEATURE if USE_REAL else 1
vec = sae.feature_direction(feat)
scorer = KeywordScorer(TERMS)
probe_scorers = {p: KeywordScorer([p]) for p in PROBES}
kw = dict(max_new_tokens=40, seed=cfg.seed)

sae_res = run_sweep(backend, vec, sae.hook_name, PROMPTS, COEFS, scorer,
                    label='sae', probe_scorers=probe_scorers, **kw)

rand = build_random_vector(vec, seed=cfg.seed)
rand_res = run_sweep(backend, rand, sae.hook_name, PROMPTS, COEFS, scorer, label='random', **kw)

pos_idx, neg_idx = label_snippets(snippets, TERMS)
md_vec = build_meandiff_direction(backend, sae.hook_name,
                                  [snippets[i] for i in pos_idx],
                                  [snippets[i] for i in neg_idx], model_name=cfg.model.name)
md_vec = match_norm(md_vec, vec)
md_res = run_sweep(backend, md_vec, sae.hook_name, PROMPTS, COEFS, scorer, label='meandiff', **kw)

results = {'sae': sae_res, 'random': rand_res, 'meandiff': md_res}

In [ ]:
# Results table
try:
    import pandas as pd
    df = pd.DataFrame(sae_res.to_dicts())
    display(df.round(3))
except ImportError:
    for row in sae_res.to_dicts():
        print(row)

In [ ]:
from scalpel.plotting import (plot_dose_response, plot_fluency,
                              plot_baseline_comparison, plot_specificity)

out = Path(OUT)
for res in results.values():
    res.write_csv(out / f'sweep_{res.label}.csv')
    res.write_json(out / f'sweep_{res.label}.json')

p_dr = plot_dose_response(sae_res, out / 'dose_response.png', concept=CONCEPT)
p_fl = plot_fluency(sae_res, out / 'fluency.png')
p_bl = plot_baseline_comparison(results, out / 'baselines.png', concept=CONCEPT)
p_sp = plot_specificity(sae_res, out / 'specificity.png', target=CONCEPT)
for p in (p_dr, p_bl, p_sp, p_fl):
    display(Image(filename=str(p)))

## Takeaways

- **Dose-response:** the SAE feature causally raises the concept, peaking at a
  fluency-limited sweet spot.
- **Random control (required):** a norm-matched random direction produces
  **zero** effect at any fluency cost — the SAE feature's effect is real and
  direction-specific.
- **Mean-difference:** a strong, competitive baseline (often matching or beating
  the SAE feature). We report this honestly rather than cherry-picking.
- **Specificity:** off-target concepts stay flat while the target rises.

Everything above is reproduced by `scalpel eval` from the command line.